## Implementing algorithm 1 from the paper. 
Computing omega points for the 3vs1 case
including the functions to check if points are in a good position 


## Todo: 
- look  out for conflicts between modules and modules_old
- Can we add function to check if 3 pts 


In [1]:
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib
from math import cos, sin, pi, atan

import numpy.linalg

## for checking if in convex hull
import scipy.spatial
from scipy.optimize import linprog

import sys 
sys.path.append('../Code/')
import modules as mod

# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py
%load_ext autoreload
%autoreload 2

## Hardcoded data

In [2]:
# here we define four critical values to check. 
# later we'll want to import these 4 at a time from a bigger set, but for now we just hardcode them.
c1 = mod.critical_val((0,0), 0)
c2 = mod.critical_val((0,1), 0)
c3 = mod.critical_val((1,0),0)
c4 = mod.critical_val((1,1),0)
CritPts = [c1,c2,c3,c4]

In [3]:
c4.parent_module

0

In [4]:
c1.coords

(0, 0)

## Convex hull functions

In [5]:
def in_hull(points, x):
    ## check if x in convex hull of points 
    ## points = array of pts in R^2 
    
    n_points = len(points)
    c = np.zeros(n_points)
    A = np.r_[points.T,np.ones((1,n_points))]
    b = np.r_[x, np.ones(1)]
    lp = linprog(c, A_eq=A, b_eq=b, )
    return lp.success

def sort_bottom_and_right(u,v,w): 
    """return the rightmost and bottommost points of u,v,w """
    
    rm = u
    if v[0]>rm[0]:
        rm = v
    if w[0] > rm[0]:
        rm = w
        
    bm = u
    if v[1]< bm[1]:
        bm = v
    if w[1] < bm[1]:
        bm = w

    return bm, rm


def check_in_quadrant_tl(x, p, q): 
    """"assume p is the bottomost pt of u,v,w
        assume q is the rightmost of u,v,w 
        return True if x is in the top left quadrant 
    """
 
    if x[0]<= q[0] and x[1]>=p[1]: 
        return True 
    else: 
        return False 
        

def check_x_push_up(x,u,v,w): ## Check-point-A and Check-point-B like in the overleaf 
    """Check if the points x,u,v,w are admissible for:
            - x pushing up
            - u,v,w pushing right 
        returns True if they are an admissible quadruple 
        and False if they are not admissible or would create superfluous omegapoints 
    """    
    p,q = sort_bottom_and_right(u,v,w)
    a = (q[0],p[1])
    
    ## First check if x is in the upper left quadrant: 
    if check_in_quadrant_tl(x,p,q): 
        ## Check if x is in convex hull
        if in_hull(np.array([u,v,w]),x):
            return False
        
        elif in_hull(np.array([u,v,w,a]),x ):
            return True
        
        else:
            return False
        
    else: 
        return True 

    
def sort_top_and_left(u,v,w): 
    """return the leftmost and topmost points of u,v,w """
        
    lm = u
    if v[0]<lm[0]:
        lm = v
    if w[0] < lm[0]:
        lm = w
        
    tm = u
    if v[1]> tm[1]:
        tm = v
    if w[1] > tm[1]:
        tm = w

    return tm, lm


def check_in_quadrant_br(x, p, q): 
    """"assume p is the topmost pt of u,v,w
        assume q is the leftmost of u,v,w 
        return True if x is in the bottom right quadrant(including the boundary)
    """
    
    if x[0]>= q[0] and x[1]<=p[1]: 
        return True 
    else: 
        return False 

    
def check_x_push_right(x,u,v,w): 
    """Check if the points x,u,v,w are admissible for:
            - x pushing right
            - u,v,w pushing up 
        returns True if they are an admissible quadruple 
        and False if they are not admissible or would create superfluous omegapoints 
    """    
    
    p,q = sort_top_and_left(u,v,w)
    a = (q[0],p[1])
    
    ## First check if x is in the upper left quadrant: 
    if check_in_quadrant_br(x,p,q): 
        ## Check if x is in convex hull
        if in_hull(np.array([u,v,w]),x):
            return False
        
        elif in_hull(np.array([u,v,w,a]),x ):
            return True  
        else: 
            return False 
    else: 
        return True 
    

## Functions

In [6]:
def check_3distinct(C):
    # this function checks that 3 out of 4 things in a set are distinct
    assert len(C) == 4 , "C is not of length 4 "
    if (len(C) - len(set(C))) > 1: 
        return False
    else: 
        return True 
    
def check_c_distinct(c,C):
    """this function checks that c, assumed to be in C, is not repeated in C"""
    C_new = C.copy()
    C_new.remove(c)
    val = True 
    for d in C_new: 
        if c.coords == d.coords: 
            val = False
    
    return val

def check_omega(CritPts,omega): 
    """ Function to be implemented later to check if the computed omegapoint is valid 
        for now: returns true all the time
    """
    
    return True

## Here begins the main loop of the algorithm

In [18]:

def algo_3vs1(CritPts): 


    assert check_3distinct(CritPts), "There is more than one repeated point in the list"

    Omega_pts = []

    for c in CritPts: 
        # Trying to find x (of the points x,w,u,v in the paper)
        if check_c_distinct(c, CritPts): 
            x = c 
        else: 
            continue ### Skips the rest of the loop if c appears multiple times in C 

        CritPts_no_x = CritPts.copy()
        CritPts_no_x.remove(x)

        for d in CritPts_no_x: 
            w = d
            CritPts_no_x_no_w  = CritPts_no_x.copy()
            CritPts_no_x_no_w.remove(w) ## lol (with a french accent), list with two elements only 
            
            if CritPts_no_x_no_w[0].coords == CritPts_no_x_no_w[1].coords:
                
                continue 
                
            u = CritPts_no_x_no_w[0]
            v = CritPts_no_x_no_w[1]

        ## Assign the value of the denominator eta and delta according to the parent module
        ## of the pairs (x,w) and (u,v)
            if x.parent_module == w.parent_module: 
                eta = 2
            else: 
                eta = 1 

            if u.parent_module == v.parent_module: 
                delta = 2
            else: 
                delta = 1 

            ## Is indentation correct? 
            if check_x_push_up(x.coords,u.coords,v.coords,w.coords):

                if u.coords[1] <= v.coords[1]: ## check if this needs to be <= or < 
                    temp_u = u
                    u = v
                    v = temp_u
                if w.coords[0] >= x.coords[0] and  w.coords[1] >= x.coords[1] : ## should this be strict inequality? , also check the coordinates 0 and 1 

                    omega = (x.coords[0],(eta/delta)*(v.coords[1]-u.coords[1])+w.coords[1])

                elif w.coords[0] <= x.coords[0] and  w.coords[1] <= x.coords[1] :

                    omega = (x.coords[0],(eta/delta)*(u.coords[1]-v.coords[1])+w.coords[1])

                else :
                    omega = (x.coords[0],(eta/delta)*(u.coords[1]-v.coords[1])+w.coords[1])
                    omega1 = (x.coords[0],(eta/delta)*(v.coords[1]-u.coords[1])+w.coords[1])

                    Omega_pts.append(omega1) ## append


                Omega_pts.append(omega) ## in any case we append omegaa

                ## print!
                #print('The omega point {} was generated from'.format(omega))
                #print('x = {}'.format(x))
                #print('w = {}'.format(w))
                #print('u = {}'.format(u))
                #print('v = {}'.format(v))
                #print('****************')



               ### Is indentation correct?  
            if check_x_push_right(x.coords,u.coords,v.coords,w.coords):

                if u.coords[0]<= v.coords[0]:
                    temp_u = u
                    u = v
                    v = temp_u
                if w.coords[0] >= x.coords[0] and w.coords[1] >= x.coords[1]: ## check < vs <= and coordinates 0,1

                    omega = ((eta/delta)*(v.coords[0]-u.coords[0])+w.coords[0], x.coords[1])

                elif w.coords[0] <= x.coords[0] and w.coords[1] <= x.coords[1]:
                    omega = ((eta/delta)*(u.coords[0]-v.coords[0])+w.coords[0], x.coords[1])

                else: 
                    omega = ((eta/delta)*(v.coords[0]-u.coords[0])+w.coords[0], x.coords[1])
                    omega1 = ((eta/delta)*(u.coords[0]-v.coords[0])+w.coords[0], x.coords[1])
                    Omega_pts.append(omega1)

                Omega_pts.append(omega)

                #print('The omega point {} was generated from'.format(omega))
                #print('x = {}'.format(x))
                #print('w = {}'.format(w))
                #print('u = {}'.format(u))
                #print('v = {}'.format(v))
                #print('****************')




    print("Computed {} omega points with duplicates".format(len(Omega_pts)))         
                
    Omega_pts = list(set(Omega_pts))   
    print("Without duplicates: {}".format(len(Omega_pts)))

    return Omega_pts




In [19]:
# here we define four critical values to check. 
# later we'll want to import these 4 at a time from a bigger set, but for now we just hardcode them.
c1 = mod.critical_val((0,0), 0)
c2 = mod.critical_val((0,1), 0)
c3 = mod.critical_val((1,0),0)
c4 = mod.critical_val((1,1),0)
CritPts = [c1,c2,c3,c4]
algo_3vs1(CritPts)

Computed 8 omega points with duplicates
Without duplicates: 4


[(0.0, 1), (1, 0.0), (2.0, 1), (1, 2.0)]

## Examples from the paper

#### example 2.11, figure 3

In [44]:

c1 = mod.critical_val((0,0), 0)
c2 = mod.critical_val((7,0), 0)
c3 = mod.critical_val((0,4),0)
c4 = mod.critical_val((7,4),0)
c5 = mod.critical_val((0,7),0)
c6 = mod.critical_val((7,7),0)
c7 = mod.critical_val((0,11),0)
c8 = mod.critical_val((7,11),0)


d1 = mod.critical_val((0,0), 1)
d2 = mod.critical_val((7,0), 1)
d3 = mod.critical_val((0,4),1)
d4 = mod.critical_val((7,4),1)
d5 = mod.critical_val((0,7),1)
d6 = mod.critical_val((7,7),1)
d7 = mod.critical_val((0,11),1)
d8 = mod.critical_val((7,11),1)




CritPts = [c1,c2,c3,c4,c5,c6,c7,c8,d1,d2,d3,d4,d5,d6,d7,d8]


## rectangle 0 

In [45]:
Pts =[]

for a in range(len(CritPts)): 
    for b in range(a+1,len(CritPts)):
        for c in range(b+1,len(CritPts)):
            for d in range(c, len(CritPts)):
                if check_3distinct([CritPts[a],CritPts[b],CritPts[c],CritPts[d]]):
                    pts = algo_3vs1([CritPts[a],CritPts[b],CritPts[c],CritPts[d]])
                    Pts +=pts
print('**********')
print("Computed {} points (with duplicates)".format(len(Pts)))

Pts = list(set(Pts))
print("Without duplicates {}".format(len(Pts)))
                    


/Users/celiahacker/opt/anaconda3/envs/celia/lib/python3.7/site-packages/ipykernel_launcher.py:9: OptimizeWarning: A_eq does not appear to be of full row rank. To improve performance, check the problem formulation for redundant equality constraints.
  if __name__ == '__main__':


Computed 4 omega points with duplicates
Without duplicates: 2
Computed 8 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 6
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 8 omega points with duplicates
Without duplicates: 8
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 11 omega points with duplicates
Without duplicates: 10
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 11 omega points with duplicates
Without duplicates: 10
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 5
Co

Computed 6 omega points with duplicates
Without duplicates: 5
Computed 12 omega points with duplicates
Without duplicates: 11
Computed 9 omega points with duplicates
Without duplicates: 8
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 9
Computed 12 omega points with duplicates
Without duplicates: 11
Computed 2 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 2 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 11 omega points with duplicates
Without duplicates: 9
Computed 6 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 9
Computed 2 omega points with duplicates
Without duplicates: 2
Computed 9 omega points with duplicates
Without duplicates: 8
C

/Users/celiahacker/opt/anaconda3/envs/celia/lib/python3.7/site-packages/ipykernel_launcher.py:9: OptimizeWarning: Solving system with option 'cholesky':True failed. It is normal for this to happen occasionally, especially as the solution is approached. However, if you see this frequently, consider setting option 'cholesky' to False.
  if __name__ == '__main__':
/Users/celiahacker/opt/anaconda3/envs/celia/lib/python3.7/site-packages/ipykernel_launcher.py:9: OptimizeWarning: Solving system with option 'sym_pos':True failed. It is normal for this to happen occasionally, especially as the solution is approached. However, if you see this frequently, consider setting option 'sym_pos' to False.
  if __name__ == '__main__':
/Users/celiahacker/opt/anaconda3/envs/celia/lib/python3.7/site-packages/ipykernel_launcher.py:9: OptimizeWarning: Solving system with option 'sym_pos':False failed. This may happen occasionally, especially as the solution is approached. However, if you see this frequently, 

Computed 12 omega points with duplicates
Without duplicates: 7
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 12 omega points with duplicates
Without duplicates: 11
Computed 6 omega points with duplicates
Without duplicates: 5
Computed 11 omega points with duplicates
Without duplicates: 10
Computed 6 omega points with duplicates
Without duplicates: 5
Computed 12 omega points with duplicates
Without duplicates: 10
Computed 12 omega points with duplicates
Without duplicates: 9
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 11 omega points with duplicates
Without duplicates: 5
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 9
Computed 11 omega points with duplicates
Without duplicates: 10
Computed 6 omega points with duplicates
Without duplicates: 6
Computed 12 omega points with duplicates
Without duplicates

...

Computed 4 omega points with duplicates
Without duplicates: 1
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 8
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 9 omega points with duplicates
Without duplicates: 8
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 8 omega points with duplicates
Without duplicates: 7
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 2 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 12 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 11 omega points with duplicates
Without duplicates: 5
Comput

Computed 11 omega points with duplicates
Without duplicates: 10
Computed 0 omega points with duplicates
Without duplicates: 0
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 6 omega points with duplicates
Without duplicates: 5
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 11 omega points with duplicates
Without duplicates: 10
Computed 12 omega points with duplicates
Without duplicates: 11
Computed 2 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 8
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 0 omega points with duplicates
Without duplicates: 0
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 6 omega points with duplicates
Without duplicates: 6
Computed 6 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 1
Co

Computed 6 omega points with duplicates
Without duplicates: 6
Computed 6 omega points with duplicates
Without duplicates: 5
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 11 omega points with duplicates
Without duplicates: 9
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 11 omega points with duplicates
Without duplicates: 6
Computed 9 omega points with duplicates
Without duplicates: 8
Computed 4 omega points with duplicates
Without duplicates: 3
Computed 10 omega points with duplicates
Without duplicates: 10
Computed 11 omega points with duplicates
Without duplicates: 8
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 2 omega points with duplicates
Without duplicates: 2
Computed 10 omega points with duplicates
Without duplicates: 5
Computed 6 omega points with duplicates
Without duplicates: 4
Co

Computed 10 omega points with duplicates
Without duplicates: 8
Computed 10 omega points with duplicates
Without duplicates: 5
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 6 omega points with duplicates
Without duplicates: 3
Computed 10 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 7
Computed 6 omega points with duplicates
Without duplicates: 2
Computed 4 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 0 omega points with duplicates
Without duplicates: 0
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 9 omega points with duplicates
Without duplicates: 7
Compu

Computed 11 omega points with duplicates
Without duplicates: 6
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 4 omega points with duplicates
Without duplicates: 2
Computed 8 omega points with duplicates
Without duplicates: 4
Computed 9 omega points with duplicates
Without duplicates: 4
Computed 11 omega points with duplicates
Without duplicates: 6
Computed 4 omega points with duplicates
Without duplicates: 1
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 9 omega points with duplicates
Without duplicates: 5
Computed 4 omega points with duplicates
Without duplicates: 2
Compu

In [46]:
print('done')

done


In [47]:
print(Pts)

[(7, 3.0), (7, 3.5), (7, 12.0), (14.0, 4), (7, 12.5), (0.0, 7), (7, 25.0), (0, 7.5), (7, -15.0), (3.5, 7), (10.5, 7), (7, 6.0), (7, 19.0), (0.0, 4), (7, 5.0), (7, 5.5), (7, 22.0), (0, -3.0), (0, 1.0), (7, -18.0), (7, 8.0), (7, 8.5), (7, 21.0), (3.5, 11), (7, -1.0), (7, -2.0), (7, 2.0), (7, 2.5), (7, 15.0), (0.0, 11), (7, -3.0), (10.5, 4), (7, 1.5), (7, 1.0), (7, 18.0), (0, 5.5), (0, 5.0), (21.0, 11), (0, 8.0), (7, 4.0), (7, 17.0), (0, -1.0), (0, -2.0), (0, 2.0), (0, 2.5), (-7.0, 4), (7, 11.0), (7, 33.0), (7, -7.0), (-7.0, 7), (7, 14.0), (7, 14.5), (21.0, 4), (7, -4.0), (7, 0.0), (7, 0.5), (7, 13.0), (14.0, 7), (0, 9.5), (0, 9.0), (7, -10.0), (21.0, 7), (3.5, 4), (10.5, 11), (7.0, 7), (7, 7.5), (7, 16.5), (0, 3.5), (7, 29.0), (0, 3.0), (-7.0, 11), (7, -11.0), (7, 10.5), (7, 10.0), (0, -4.0), (0, 0.0), (0, 0.5), (7, 9.0), (14.0, 11), (7, 9.5), (0, -10.0), (7, 26.0), (0, -7.0), (7, -1.5)]


## Further analysis: 
    1. plot omega pts and critical points 
    2. Can we determine which ones a